# SEECS, National University of Sciences and Technology
## EE348: Knowledge Representation and Reasoning in AI (BSAI-2024A)
### Assignment 1: University Research Lab Access Reasoning System
**Due Date:** 15 OCT 5 PM  
**Student Name:** Sikandar Hussain  
**Course Instructor:** Dr. Saqib Nawaz | **Lab Engineer:** Syed Muhammad Ali Musa  

---

### Scenario Overview
A university uses a finite knowledge base to determine which students qualify for a research lab.
In this notebook, we build:
1. A **Finite-Model Checker** to evaluate quantified First-Order Logic (FOL) formulas over finite domain models.
2. A **Clausal CNF Converter & Unifier** for resolution theorem proving.
3. An **Automated Forward-Chaining Inference Engine** (Horn-style rules) computing new facts to a fixed point.
4. A **Goal-Directed Backward-Chaining Engine** producing subgoal resolution traces and comparative analysis.

> **Knowledge Representation Protocol:**  
> - Closed World Assumption (CWA) for finite-model checking: facts not listed are evaluated as `False`.  
> - For rule-based deductive queries: queries not proven are reported as `Not Derivable` (distinguishing negation as failure from explicit negation).

## Knowledge Base Initialization
We define the exact initial sets and predicate functions provided in the assignment specification:
- `domain = {"Ali", "Sara", "Bilal", "KRR", "AI"}`
- `students = {"Ali", "Sara", "Bilal"}`
- `courses = {"KRR", "AI"}`
- `enrolled = {("Ali", "KRR"), ("Sara", "AI"), ("Bilal", "KRR")}`
- `passed = {("Ali", "KRR"), ("Bilal", "KRR")}`
- `orientation_complete = {"Ali", "Sara"}`

In [1]:
# Exact initial domain and relations
domain = {"Ali", "Sara", "Bilal", "KRR", "AI"}
students = {"Ali", "Sara", "Bilal"}
courses = {"KRR", "AI"}
enrolled = {("Ali", "KRR"), ("Sara", "AI"), ("Bilal", "KRR")}
passed = {("Ali", "KRR"), ("Bilal", "KRR")}
orientation_complete = {"Ali", "Sara"}

# Predicate functions
def Student(x):
    return x in students

def Course(x):
    return x in courses

def Enrolled(x, c):
    return (x, c) in enrolled

def Passed(x, c):
    return (x, c) in passed

def OrientationComplete(x):
    return x in orientation_complete

print("Knowledge Base initialized successfully:")
print(f"  Domain:               {domain}")
print(f"  Students:             {students}")
print(f"  Courses:              {courses}")
print(f"  Enrolled:             {enrolled}")
print(f"  Passed:               {passed}")
print(f"  Orientation Complete: {orientation_complete}")

Knowledge Base initialized successfully:
  Domain:               {'KRR', 'Sara', 'Ali', 'AI', 'Bilal'}
  Students:             {'Ali', 'Sara', 'Bilal'}
  Courses:              {'KRR', 'AI'}
  Enrolled:             {('Bilal', 'KRR'), ('Sara', 'AI'), ('Ali', 'KRR')}
  Passed:               {('Bilal', 'KRR'), ('Ali', 'KRR')}
  Orientation Complete: {'Ali', 'Sara'}


## Task 1. First-Order Logic (FOL) and Finite-Model Checking

### a) English-to-FOL Formal Translation

1. **Formula (i):** *"Every student is enrolled in at least one course."*
   $$\forall x \ (\text{Student}(x) \to \exists c \ (\text{Course}(c) \land \text{Enrolled}(x, c)))$$

2. **Formula (ii):** *"Some course has every student enrolled."*
   $$\exists c \ (\text{Course}(c) \land \forall x \ (\text{Student}(x) \to \text{Enrolled}(x, c)))$$

3. **Formula (iii):** *"Every student enrolled in KRR has passed KRR."*
   $$\forall x \ ((\text{Student}(x) \land \text{Enrolled}(x, \text{KRR})) \to \text{Passed}(x, \text{KRR}))$$

In [2]:
# Task 1 b) Implement and evaluate all three formulas using all() and any()

# Formula (i): Every student is enrolled in at least one course
# ∀x (Student(x) → ∃c (Course(c) ∧ Enrolled(x, c)))
formula_1 = all(
    any(Course(c) and Enrolled(s, c) for c in domain)
    for s in domain if Student(s)
)

# Formula (ii): Some course has every student enrolled
# ∃c (Course(c) ∧ ∀x (Student(x) → Enrolled(x, c)))
formula_2 = any(
    Course(c) and all(Enrolled(s, c) for s in domain if Student(s))
    for c in domain
)

# Formula (iii): Every student enrolled in KRR has passed KRR
# ∀x ((Student(x) ∧ Enrolled(x, "KRR")) → Passed(x, "KRR"))
formula_3 = all(
    (not (Student(s) and Enrolled(s, "KRR"))) or Passed(s, "KRR")
    for s in domain
)

print("Task 1 b) Finite-Model Checking Results:")
print("-" * 50)
print(f"  Formula (i)   [Every student enrolled in >= 1 course]:  {formula_1}")
print(f"  Formula (ii)  [Some course has every student enrolled]: {formula_2}")
print(f"  Formula (iii) [Every KRR-enrolled student passed KRR]:  {formula_3}")
print("-" * 50)

Task 1 b) Finite-Model Checking Results:
--------------------------------------------------
  Formula (i)   [Every student enrolled in >= 1 course]:  True
  Formula (ii)  [Some course has every student enrolled]: False
  Formula (iii) [Every KRR-enrolled student passed KRR]:  True
--------------------------------------------------


In [3]:
# Task 1 c) Identify witnesses and counterexamples

print("Task 1 c) Witnesses and Counterexamples Analysis:")
print("-" * 65)

# Witness for Formula (i) [True existential claim per student]
print("1. Witnesses for Formula (i) (True claim):")
for s in sorted(students):
    student_courses = [c for c in courses if Enrolled(s, c)]
    print(f"   - Student '{s}' has enrollment witness: course '{student_courses[0]}'")

print("\n2. Counterexample Analysis for Formula (ii) (False claim):")
# Check each course to show why no course satisfies the universal condition
for c in sorted(courses):
    missing_students = [s for s in sorted(students) if not Enrolled(s, c)]
    print(f"   - For Course '{c}': NOT all students enrolled. Counterexample student(s): {missing_students}")

print("\n3. Verification for Formula (iii) (True universal claim):")
krr_students = [s for s in sorted(students) if Enrolled(s, "KRR")]
for s in krr_students:
    print(f"   - Student '{s}' is enrolled in KRR and Passed(s, KRR) = {Passed(s, 'KRR')} (Satisfied)")

Task 1 c) Witnesses and Counterexamples Analysis:
-----------------------------------------------------------------
1. Witnesses for Formula (i) (True claim):
   - Student 'Ali' has enrollment witness: course 'KRR'
   - Student 'Bilal' has enrollment witness: course 'KRR'
   - Student 'Sara' has enrollment witness: course 'AI'

2. Counterexample Analysis for Formula (ii) (False claim):
   - For Course 'AI': NOT all students enrolled. Counterexample student(s): ['Ali', 'Bilal']
   - For Course 'KRR': NOT all students enrolled. Counterexample student(s): ['Sara']

3. Verification for Formula (iii) (True universal claim):
   - Student 'Ali' is enrolled in KRR and Passed(s, KRR) = True (Satisfied)
   - Student 'Bilal' is enrolled in KRR and Passed(s, KRR) = True (Satisfied)


### Task 1 c) Short Explanation of Witnesses and Counterexamples
> **Analysis:**  
> - **Witness for Formula (i):** The universal claim holds because every student has an existential course witness in the model: `Ali` is enrolled in `KRR`, `Sara` is enrolled in `AI`, and `Bilal` is enrolled in `KRR`.  
> - **Counterexample for Formula (ii):** The existential claim is `False` because neither course enrolls all three students:
>   - For course `KRR`, `Sara` serves as a counterexample because $\text{Enrolled}(\text{Sara}, \text{KRR})$ is false (she is enrolled in `AI`).  
>   - For course `AI`, both `Ali` and `Bilal` serve as counterexamples because neither is enrolled in `AI`.  
> - **Formula (iii):** Evaluates to `True` because the antecedent $(\text{Student}(x) \land \text{Enrolled}(x, \text{KRR}))$ holds only for $x \in \{\text{Ali}, \text{Bilal}\}$, and both have explicitly passed `KRR`. For all other domain elements, the material conditional is vacuously true.

## Task 2. Conjunctive Normal Form (CNF) and Unification

### a) Clausal CNF Conversion
We convert the rule:
$$\text{KRRLearner}(x) \land \text{Passed}(x, \text{KRR}) \to \text{EligibleForLab}(x)$$

**Step 1: Implication Elimination** ($P \to Q \equiv \neg P \lor Q$):
$$\neg (\text{KRRLearner}(x) \land \text{Passed}(x, \text{KRR})) \lor \text{EligibleForLab}(x)$$

**Step 2: De Morgan's Law Application** ($\neg(A \land B) \equiv \neg A \lor \neg B$):
$$\neg \text{KRRLearner}(x) \lor \neg \text{Passed}(x, \text{KRR}) \lor \text{EligibleForLab}(x)$$

**Step 3: Clausal Representation:**
$$\text{Clause} = \{\neg \text{KRRLearner}(x), \neg \text{Passed}(x, \text{KRR}), \text{EligibleForLab}(x)\}$$

---

### b) Unification and Substitution Application
1. **Target Unification:** Unify literal $L_1 = \text{KRRLearner}(x)$ with $L_2 = \text{KRRLearner}(\text{Ali})$.
   - Both share the same predicate symbol (`KRRLearner`) of arity 1.
   - Argument 1 of $L_1$ is variable $x$; argument 1 of $L_2$ is constant `Ali`.
   - The **Most General Unifier (MGU)** is:
     $$\theta = \{x / \text{Ali}\}$$

2. **Applying Substitution to $\text{EligibleForLab}(x)$:**
   $$\text{EligibleForLab}(x)\theta = \text{EligibleForLab}(\text{Ali})$$

In [4]:
# Task 2 Implementation: Programmatic Clausal Representation & Unification

def unify_atoms(atom1, atom2):
    # Unifies two atomic formulas of format (predicate, *args)
    pred1, args1 = atom1[0], atom1[1:]
    pred2, args2 = atom2[0], atom2[1:]
    
    if pred1 != pred2 or len(args1) != len(args2):
        return None  # Predicate mismatch or arity mismatch
    
    subst = {}
    for a1, a2 in zip(args1, args2):
        if a1 == a2:
            continue
        elif a1.islower():  # a1 is a variable (e.g., 'x')
            subst[a1] = a2
        elif a2.islower():  # a2 is a variable
            subst[a2] = a1
        else:
            return None  # Constant clash (e.g., Ali != Sara)
    return subst

def apply_subst(atom, subst):
    pred = atom[0]
    args = [subst.get(arg, arg) for arg in atom[1:]]
    return (pred, *args)

# Test unification for Task 2 b)
atom1 = ("KRRLearner", "x")
atom2 = ("KRRLearner", "Ali")
target_atom = ("EligibleForLab", "x")

mgu = unify_atoms(atom1, atom2)
substituted_result = apply_subst(target_atom, mgu)

print("Task 2 b) Unification & Substitution Results:")
print("-" * 55)
print(f"  Atom 1:               {atom1[0]}({atom1[1]})")
print(f"  Atom 2:               {atom2[0]}({atom2[1]})")
print(f"  Computed MGU (θ):     {mgu}")
print(f"  Target Literal:       {target_atom[0]}({target_atom[1]})")
print(f"  Substituted Literal:  {substituted_result[0]}({substituted_result[1]})")
print("-" * 55)

Task 2 b) Unification & Substitution Results:
-------------------------------------------------------
  Atom 1:               KRRLearner(x)
  Atom 2:               KRRLearner(Ali)
  Computed MGU (θ):     {'x': 'Ali'}
  Target Literal:       EligibleForLab(x)
  Substituted Literal:  EligibleForLab(Ali)
-------------------------------------------------------


## Task 3. Automated Reasoning: Forward Chaining
We implement a forward-chaining inference engine for Horn-style rules:
- **R1:** $\text{Student}(x) \land \text{Enrolled}(x, \text{KRR}) \to \text{KRRLearner}(x)$
- **R2:** $\text{KRRLearner}(x) \land \text{Passed}(x, \text{KRR}) \to \text{EligibleForLab}(x)$
- **R3:** $\text{EligibleForLab}(x) \land \text{OrientationComplete}(x) \to \text{AuthorizedForLab}(x)$

### a) Python Structures for Facts and Rules
- **Facts:** Stored as tuples of relation and arguments: `("Predicate", arg1, ...)`
- **Rules:** Stored as structured dictionaries containing:
  - `name`: Rule identifier (`R1`, `R2`, `R3`)
  - `antecedents`: List of required premises with variable `$x`
  - `consequent`: Target conclusion with variable `$x`

In [5]:
# Task 3 a) Knowledge Base Representation

# Base explicit ground facts
initial_facts = set()

for s in students:
    initial_facts.add(("Student", s))

for c in courses:
    initial_facts.add(("Course", c))

for s, c in enrolled:
    initial_facts.add(("Enrolled", s, c))

for s, c in passed:
    initial_facts.add(("Passed", s, c))

for s in orientation_complete:
    initial_facts.add(("OrientationComplete", s))

# Horn Rules
rules = [
    {
        "name": "R1",
        "description": "Student(x) AND Enrolled(x,KRR) -> KRRLearner(x)",
        "antecedents": [("Student", "x"), ("Enrolled", "x", "KRR")],
        "consequent": ("KRRLearner", "x")
    },
    {
        "name": "R2",
        "description": "KRRLearner(x) AND Passed(x,KRR) -> EligibleForLab(x)",
        "antecedents": [("KRRLearner", "x"), ("Passed", "x", "KRR")],
        "consequent": ("EligibleForLab", "x")
    },
    {
        "name": "R3",
        "description": "EligibleForLab(x) AND OrientationComplete(x) -> AuthorizedForLab(x)",
        "antecedents": [("EligibleForLab", "x"), ("OrientationComplete", "x")],
        "consequent": ("AuthorizedForLab", "x")
    }
]

print(f"Total Base Facts Loaded: {len(initial_facts)}")
print("Sample Base Facts:", sorted(list(initial_facts))[:6])
print(f"Total Rules Loaded:     {len(rules)}")

Total Base Facts Loaded: 12
Sample Base Facts: [('Course', 'AI'), ('Course', 'KRR'), ('Enrolled', 'Ali', 'KRR'), ('Enrolled', 'Bilal', 'KRR'), ('Enrolled', 'Sara', 'AI'), ('OrientationComplete', 'Ali')]
Total Rules Loaded:     3


In [6]:
# Task 3 b) Forward-Chaining Algorithm to Fixed Point

def forward_chain(facts, rules, domain_candidates):
    # Derives all reachable consequences until no new facts can be added (fixed point).
    known_facts = set(facts)
    derivation_trace = []
    iteration = 1
    
    while True:
        new_facts_this_round = []
        
        for rule in rules:
            rule_name = rule["name"]
            
            # Match variable 'x' across all candidate entities in domain
            for candidate in sorted(domain_candidates):
                subst = {"x": candidate}
                
                # Ground all antecedents with candidate
                ground_antecedents = [apply_subst(ant, subst) for ant in rule["antecedents"]]
                
                # Check if all ground antecedents are currently in known_facts
                if all(ant in known_facts for ant in ground_antecedents):
                    ground_consequent = apply_subst(rule["consequent"], subst)
                    
                    if ground_consequent not in known_facts and ground_consequent not in [f[0] for f in new_facts_this_round]:
                        new_facts_this_round.append((ground_consequent, rule_name, candidate))
        
        if not new_facts_this_round:
            break  # Fixed point reached
            
        print(f"\n--- Forward-Chaining Iteration {iteration} ---")
        for fact, r_name, cand in new_facts_this_round:
            known_facts.add(fact)
            fact_str = f"{fact[0]}({', '.join(fact[1:])})"
            derivation_trace.append((fact, r_name, cand))
            print(f"  [DERIVED] {fact_str:<25} via Rule {r_name} (x = {cand})")
            
        iteration += 1
        
    return known_facts, derivation_trace

print("Executing Task 3 b) Forward Chaining to Fixed Point:")
final_kb_facts, trace = forward_chain(initial_facts, rules, students)
print("\nFixed point reached! No more new facts can be derived.")
print(f"Total Facts in Final Knowledge Base: {len(final_kb_facts)}")

Executing Task 3 b) Forward Chaining to Fixed Point:

--- Forward-Chaining Iteration 1 ---
  [DERIVED] KRRLearner(Ali)           via Rule R1 (x = Ali)
  [DERIVED] KRRLearner(Bilal)         via Rule R1 (x = Bilal)

--- Forward-Chaining Iteration 2 ---
  [DERIVED] EligibleForLab(Ali)       via Rule R2 (x = Ali)
  [DERIVED] EligibleForLab(Bilal)     via Rule R2 (x = Bilal)

--- Forward-Chaining Iteration 3 ---
  [DERIVED] AuthorizedForLab(Ali)     via Rule R3 (x = Ali)

Fixed point reached! No more new facts can be derived.
Total Facts in Final Knowledge Base: 17


In [7]:
# Task 3 c) Query AuthorizedForLab for Ali, Bilal, and Sara

candidates_to_query = ["Ali", "Bilal", "Sara"]
print("Task 3 c) Evaluation of Lab Authorization Queries:")
print("=" * 70)

for person in candidates_to_query:
    query = ("AuthorizedForLab", person)
    query_str = f"AuthorizedForLab({person})"
    
    if query in final_kb_facts:
        print(f"Query: {query_str:<25} -> DERIVABLE (True)")
    else:
        print(f"Query: {query_str:<25} -> NOT DERIVABLE (Not Proven)")

print("=" * 70)

Task 3 c) Evaluation of Lab Authorization Queries:
Query: AuthorizedForLab(Ali)     -> DERIVABLE (True)
Query: AuthorizedForLab(Bilal)   -> NOT DERIVABLE (Not Proven)
Query: AuthorizedForLab(Sara)    -> NOT DERIVABLE (Not Proven)


### Task 3 c) Detailed Explanations for Query Outcomes
> **Derivation Analysis:**  
> 1. **`AuthorizedForLab(Ali)` $\to$ DERIVABLE:**  
>    - Base facts: `Student(Ali)`, `Enrolled(Ali, KRR)`, `Passed(Ali, KRR)`, and `OrientationComplete(Ali)`.  
>    - Step 1: `R1` fires with $x = \text{Ali} \implies \text{KRRLearner}(\text{Ali})$ is derived.  
>    - Step 2: `R2` fires with $\text{KRRLearner}(\text{Ali})$ and $\text{Passed}(\text{Ali}, \text{KRR}) \implies \text{EligibleForLab}(\text{Ali})$ is derived.  
>    - Step 3: `R3` fires with $\text{EligibleForLab}(\text{Ali})$ and $\text{OrientationComplete}(\text{Ali}) \implies \text{AuthorizedForLab}(\text{Ali})$ is successfully derived.  
> 
> 2. **`AuthorizedForLab(Bilal)` $\to$ NOT DERIVABLE:**  
>    - Base facts: `Student(Bilal)`, `Enrolled(Bilal, KRR)`, and `Passed(Bilal, KRR)`.  
>    - `R1` and `R2` fire, successfully deriving $\text{KRRLearner}(\text{Bilal})$ and $\text{EligibleForLab}(\text{Bilal})$.  
>    - However, $\text{OrientationComplete}(\text{Bilal})$ is **missing** from the knowledge base. Because the second conjunct of `R3` cannot be established, `R3` fails to fire. Under the open-query semantics, we conclude **Not Derivable**.  
> 
> 3. **`AuthorizedForLab(Sara)` $\to$ NOT DERIVABLE:**  
>    - Base facts: `Student(Sara)` and `OrientationComplete(Sara)`, but she is enrolled in `AI`, **not `KRR`** ($\text{Enrolled}(\text{Sara}, \text{KRR})$ is false).  
>    - Therefore, `R1` cannot fire for `Sara`, precluding the derivation of $\text{KRRLearner}(\text{Sara})$, which in turn blocks $\text{EligibleForLab}(\text{Sara})$ and $\text{AuthorizedForLab}(\text{Sara})$.

## Task 4. Backward Reasoning and Comparative Interpretation

### a) Backward Chaining Implementation & Subgoal Proof Tree
Backward chaining starts at the target goal and recursively queries whether:
1. The goal is an explicit ground fact in the base knowledge base, OR
2. A rule exists whose consequent unifies with the goal, prompting recursive verification of all its ground antecedents as subgoals.

In [8]:
# Task 4 a) Backward Chaining Engine with Subgoal Tracing

def backward_chain(goal, facts, rules, depth=0):
    indent = "  " * depth
    goal_str = f"{goal[0]}({', '.join(goal[1:])})"
    print(f"{indent}--> Checking Subgoal: {goal_str}")
    
    # Base Case 1: Subgoal is directly in the base facts
    if goal in facts:
        print(f"{indent}    [SUCCESS] Found as direct ground fact in KB.")
        return True
    
    # Case 2: Find rules that can derive this goal
    matching_rules = [r for r in rules if r["consequent"][0] == goal[0]]
    if not matching_rules:
        print(f"{indent}    [FAILURE] No rule derives predicate '{goal[0]}'.")
        return False
        
    for rule in matching_rules:
        rule_name = rule["name"]
        var_binding = goal[1]  # Binds variable x to the query argument (e.g. 'Ali')
        subst = {"x": var_binding}
        
        ground_antecedents = [apply_subst(ant, subst) for ant in rule["antecedents"]]
        print(f"{indent}    [RULE APPLIED] Rule {rule_name}: {rule['description']}")
        print(f"{indent}    Requires subgoals: {[f'{a[0]}({a[1]})' for a in ground_antecedents]}")
        
        # Recursively check each antecedent
        all_subgoals_met = True
        for ant in ground_antecedents:
            if not backward_chain(ant, facts, rules, depth + 2):
                all_subgoals_met = False
                break
                
        if all_subgoals_met:
            print(f"{indent}    [SUCCESS] All subgoals for {rule_name} verified!")
            return True
            
    print(f"{indent}    [FAILURE] Could not satisfy subgoals for '{goal_str}'.")
    return False

print("Task 4 a) Backward-Chaining Trace for Query: AuthorizedForLab(Ali):")
print("=" * 70)
ali_goal = ("AuthorizedForLab", "Ali")
ali_result = backward_chain(ali_goal, initial_facts, rules)
print("=" * 70)
print(f"Final Proof Outcome for AuthorizedForLab(Ali): {ali_result}")

Task 4 a) Backward-Chaining Trace for Query: AuthorizedForLab(Ali):
--> Checking Subgoal: AuthorizedForLab(Ali)
    [RULE APPLIED] Rule R3: EligibleForLab(x) AND OrientationComplete(x) -> AuthorizedForLab(x)
    Requires subgoals: ['EligibleForLab(Ali)', 'OrientationComplete(Ali)']
    --> Checking Subgoal: EligibleForLab(Ali)
        [RULE APPLIED] Rule R2: KRRLearner(x) AND Passed(x,KRR) -> EligibleForLab(x)
        Requires subgoals: ['KRRLearner(Ali)', 'Passed(Ali)']
        --> Checking Subgoal: KRRLearner(Ali)
            [RULE APPLIED] Rule R1: Student(x) AND Enrolled(x,KRR) -> KRRLearner(x)
            Requires subgoals: ['Student(Ali)', 'Enrolled(Ali)']
            --> Checking Subgoal: Student(Ali)
                [SUCCESS] Found as direct ground fact in KB.
            --> Checking Subgoal: Enrolled(Ali, KRR)
                [SUCCESS] Found as direct ground fact in KB.
            [SUCCESS] All subgoals for R1 verified!
        --> Checking Subgoal: Passed(Ali, KRR)
        

In [9]:
# Test backward chaining on Bilal to illustrate goal-directed failure
print("Backward-Chaining Trace for Query: AuthorizedForLab(Bilal) [To illustrate failure trace]:")
print("=" * 70)
bilal_goal = ("AuthorizedForLab", "Bilal")
bilal_result = backward_chain(bilal_goal, initial_facts, rules)
print("=" * 70)
print(f"Final Proof Outcome for AuthorizedForLab(Bilal): {bilal_result}")

Backward-Chaining Trace for Query: AuthorizedForLab(Bilal) [To illustrate failure trace]:
--> Checking Subgoal: AuthorizedForLab(Bilal)
    [RULE APPLIED] Rule R3: EligibleForLab(x) AND OrientationComplete(x) -> AuthorizedForLab(x)
    Requires subgoals: ['EligibleForLab(Bilal)', 'OrientationComplete(Bilal)']
    --> Checking Subgoal: EligibleForLab(Bilal)
        [RULE APPLIED] Rule R2: KRRLearner(x) AND Passed(x,KRR) -> EligibleForLab(x)
        Requires subgoals: ['KRRLearner(Bilal)', 'Passed(Bilal)']
        --> Checking Subgoal: KRRLearner(Bilal)
            [RULE APPLIED] Rule R1: Student(x) AND Enrolled(x,KRR) -> KRRLearner(x)
            Requires subgoals: ['Student(Bilal)', 'Enrolled(Bilal)']
            --> Checking Subgoal: Student(Bilal)
                [SUCCESS] Found as direct ground fact in KB.
            --> Checking Subgoal: Enrolled(Bilal, KRR)
                [SUCCESS] Found as direct ground fact in KB.
            [SUCCESS] All subgoals for R1 verified!
        -->

### Task 4 b) Explanation: Backward Chaining vs. Forward Chaining
> **Comparative Analysis (1 Mark):**  
> 1. **Inference Direction:** **Forward chaining** is data-driven (bottom-up), starting from initial base facts and applying all possible rules to derive every reachable fact until a fixed point is attained. In contrast, **backward chaining** is goal-directed (top-down), starting from a candidate hypothesis (e.g., `AuthorizedForLab(Ali)`) and working in reverse to identify subgoals.  
> 2. **Computational Scope:** In this knowledge base, forward chaining exhaustively computed lab eligibility for all students (deriving intermediate facts for `Bilal` as well as `Ali`), whereas backward chaining evaluated strictly the necessary subgoals required to prove `Ali`.  
> 3. **Application Fit:** Backward chaining is ideal when answering specific queries on demand without spending resources deriving irrelevant facts, whereas forward chaining is preferred for pre-computing comprehensive consequences when base data is updated.